# Modelling: anomaly detection ensemble

This notebook fits four anomaly detection methods to Calgary watershed
water quality data and combines them via majority-vote ensemble:

1. **Isolation Forest** (n_estimators=200, contamination=0.05)
2. **Local Outlier Factor** (LOF)
3. **Z-score thresholding** (threshold=3.0)
4. **Statistical detector** (mean +/- 3 std)

We compare individual detection rates and the ensemble output.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, '..')
from src.data_loader import load_and_prepare, KEY_PARAMETERS
from src.model import (
    IsolationForestDetector,
    LOFDetector,
    StatisticalDetector,
    ZScoreDetector,
    WaterQualityAnomalyDetector,
    detect_anomalies,
)

pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Load feature-engineered data

In [ ]:
long_df, wide_df = load_and_prepare(limit=50000)

# Select numeric feature columns (exclude metadata)
exclude = {'sample_site', 'sample_date', 'year', 'month'}
feature_cols = [
    c for c in wide_df.columns
    if c not in exclude and wide_df[c].dtype in ('float64', 'int64')
]

print(f'Wide-form rows: {len(wide_df):,}')
print(f'Feature columns: {len(feature_cols)}')

## 2. Isolation Forest

Isolation Forest isolates anomalies by randomly selecting a feature and a
split value. Anomalies require fewer splits to isolate, yielding shorter
average path lengths.

In [ ]:
X = wide_df[feature_cols].values

iso_detector = IsolationForestDetector(contamination=0.05, random_state=42)
iso_labels = iso_detector.fit_predict(X)
iso_scores = iso_detector.decision_scores(X)

print(f'Isolation Forest anomalies: {iso_labels.sum()} / {len(iso_labels)} ({iso_labels.mean()*100:.1f}%)')

In [ ]:
fig = px.histogram(x=iso_scores, nbins=60, title='Isolation Forest decision scores',
                   labels={'x': 'Decision score', 'y': 'Count'},
                   color_discrete_sequence=['darkorange'])
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Threshold')
fig.update_layout(height=350)
fig.show()

## 3. Local Outlier Factor (LOF)

LOF measures the local density deviation of each sample relative to its
neighbours. Points in low-density regions receive high outlier scores.

In [ ]:
lof_detector = LOFDetector(contamination=0.05, n_neighbors=20)
lof_labels = lof_detector.fit_predict(X)
lof_scores = lof_detector.negative_outlier_factors(X)

print(f'LOF anomalies: {lof_labels.sum()} / {len(lof_labels)} ({lof_labels.mean()*100:.1f}%)')

In [ ]:
fig = px.histogram(x=lof_scores, nbins=60, title='LOF negative outlier factor scores',
                   labels={'x': 'Negative outlier factor', 'y': 'Count'},
                   color_discrete_sequence=['mediumseagreen'])
fig.update_layout(height=350)
fig.show()

## 4. Z-score thresholding

A sample is flagged when any feature exceeds a z-score of 3.0.

In [ ]:
zscore_det = ZScoreDetector(threshold=3.0)
zscore_labels = zscore_det.fit_predict(X)
z_matrix = ZScoreDetector.compute_zscores(X)

print(f'Z-score anomalies: {zscore_labels.sum()} / {len(zscore_labels)} ({zscore_labels.mean()*100:.1f}%)')
print(f'Max |z| per sample -- median: {np.nanmax(np.abs(z_matrix), axis=1).mean():.2f}')

In [ ]:
stat_det = StatisticalDetector(n_std=3.0)
stat_labels = stat_det.fit_predict(X)
max_z = stat_det.max_z_per_sample(X)

print(f'Statistical detector anomalies: {stat_labels.sum()} / {len(stat_labels)} ({stat_labels.mean()*100:.1f}%)')

## 5. Ensemble voting

The `WaterQualityAnomalyDetector` combines all four methods. A sample is
labelled anomalous if the mean of the four binary votes reaches the
threshold (default 0.5 -- i.e., at least 2 out of 4 agree).

In [ ]:
detector, results, anomaly_summary = detect_anomalies(
    wide_df, feature_cols, contamination=0.05, ensemble_threshold=0.5,
)

print(f'Ensemble anomalies: {results["anomaly"].sum()} / {len(results["anomaly"])} '
      f'({results["anomaly"].mean()*100:.1f}%)')
print(f'\nAnomaly summary rows: {len(anomaly_summary)}')
anomaly_summary.head(10)

## 6. Compare detection rates

How much do the individual detectors agree?

In [ ]:
comparison = pd.DataFrame({
    'Method': ['Isolation Forest', 'LOF', 'Z-score', 'Statistical', 'Ensemble'],
    'Anomalies': [
        results['isolation_forest'].sum(),
        results['lof'].sum(),
        results['zscore'].sum(),
        results['statistical'].sum(),
        results['anomaly'].sum(),
    ],
    'Rate (%)': [
        round(results['isolation_forest'].mean() * 100, 2),
        round(results['lof'].mean() * 100, 2),
        round(results['zscore'].mean() * 100, 2),
        round(results['statistical'].mean() * 100, 2),
        round(results['anomaly'].mean() * 100, 2),
    ],
})

fig = px.bar(comparison, x='Method', y='Anomalies', text='Rate (%)',
             title='Detection rate comparison across methods',
             color='Method', color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition='outside')
fig.update_layout(height=400, showlegend=False)
fig.show()

comparison

In [ ]:
# Agreement matrix: how often do pairs of detectors both flag the same sample?
methods = ['isolation_forest', 'lof', 'zscore', 'statistical']
agreement = pd.DataFrame(index=methods, columns=methods, dtype=float)
for m1 in methods:
    for m2 in methods:
        both = ((results[m1] == 1) & (results[m2] == 1)).sum()
        either = ((results[m1] == 1) | (results[m2] == 1)).sum()
        agreement.loc[m1, m2] = round(both / max(either, 1), 3)

fig = px.imshow(agreement.astype(float), text_auto='.2f',
                title='Jaccard agreement between detectors',
                color_continuous_scale='YlOrRd')
fig.update_layout(height=400, width=500)
fig.show()

## 7. Save the ensemble model

In [ ]:
path = detector.save('water_quality_anomaly_model.joblib')
print(f'Model saved to {path}')

## Summary

- Isolation Forest (200 trees, contamination=0.05) and LOF flag roughly 5% of readings.
- Z-score and statistical detectors may flag more or fewer samples depending on data skewness.
- The ensemble (majority vote) provides a balanced detection rate.
- Agreement analysis shows which methods overlap the most.

Detailed evaluation of the detected anomalies continues in `04_evaluation.ipynb`.